<a href="https://colab.research.google.com/github/DanishShah619/git_agent/blob/main/git_pages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install requests beautifulsoup4 tqdm

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Define the output directory in Google Drive
output_dir = '/content/drive/MyDrive/rag_git/git_scraper_results'

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Google Drive mounted. Output directory created at: {output_dir}")

The Google Drive is mounted, and the output directory is ready. Now I will modify the `scrape_all` function to generate the Markdown file.

In [2]:
import requests
from bs4 import BeautifulSoup
import json
import time
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# -----------------------------
# 1. Seed URLs (add/modify here)
# -----------------------------
URLS = [
    "https://docs.github.com/en/get-started/using-git/about-git",
    "https://docs.github.com/en/get-started/learning-to-code/getting-started-with-git",
    "https://docs.github.com/en/get-started/using-git",
    "https://docs.github.com/en/repositories/configuring-branches-and-merges-in-your-repository/managing-branches-in-your-repository/about-branches",
    "https://docs.github.com/en/repositories/configuring-branches-and-merges-in-your-repository/managing-branches-in-your-repository",
    "https://docs.github.com/en/get-started/getting-started-with-git/git-workflows",
    "https://docs.github.com/en/get-started/git-basics/git-cheatsheet",
    "https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/about-pull-requests",
    "https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/creating-a-pull-request",
    "https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/merging-a-pull-request",
    "https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/reviewing-changes-in-pull-requests",
    "https://docs.github.com/en/get-started/using-git/resolving-merge-conflicts-after-a-git-rebase",
    "https://docs.github.com/en/get-started/using-git/dealing-with-non-fast-forward-errors",
    "https://docs.github.com/en/repositories/creating-and-managing-repositories/about-repositories",
    "https://docs.github.com/en/repositories/creating-and-managing-repositories/cloning-a-repository",
    "https://docs.github.com/en/repositories/creating-and-managing-repositories/creating-a-new-repository",
    "https://docs.github.com/en/repositories/creating-and-managing-repositories/forking-a-repository",
    "https://docs.github.com/en/get-started/quickstart/contributing-to-projects",
    "https://docs.github.com/en/get-started",
    "https://docs.github.com/en/github/getting-started-with-github/git-and-github-learning-resources"
]

# -----------------------------
# 2. Extract main content
# -----------------------------
def extract_content(url):
    try:
        res = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(res.text, "html.parser")

        # GitHub Docs main content container
        article = soup.find("main")

        if not article:
            return None

        content_blocks = []
        current_section = "intro"

        for tag in article.find_all(["h1", "h2", "h3", "p", "pre", "code", "ul", "ol"]):

            if tag.name in ["h1", "h2", "h3"]:
                current_section = tag.get_text(strip=True)

            elif tag.name == "p":
                content_blocks.append({
                    "type": "text",
                    "section": current_section,
                    "content": tag.get_text(strip=True)
                })

            elif tag.name in ["pre", "code"]:
                content_blocks.append({
                    "type": "code",
                    "section": current_section,
                    "content": tag.get_text("\n", strip=True)
                })

            elif tag.name in ["ul", "ol"]:
                items = [li.get_text(strip=True) for li in tag.find_all("li")]
                content_blocks.append({
                    "type": "list",
                    "section": current_section,
                    "content": items
                })

        title = soup.find("h1")
        title = title.get_text(strip=True) if title else "Untitled"

        return {
            "url": url,
            "title": title,
            "content": content_blocks
        }

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None


# -----------------------------
# 3. Run scraper
# -----------------------------
def scrape_all():
    results = []

    for url in tqdm(URLS):
        data = extract_content(url)
        if data:
            results.append(data)

        time.sleep(0.5)  # polite scraping

    with open("github_docs.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    scrape_all()

100%|██████████| 20/20 [00:14<00:00,  1.43it/s]
